# AZ Financial Spend Intelligence — Data Quality Investigation
Scope: `invoices_2025.csv` × `vendor_master.csv`

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
})

DATA = '../data/'
inv_raw = pd.read_csv(DATA + 'invoices_2025.csv', dtype=str, keep_default_na=False)
vnd     = pd.read_csv(DATA + 'vendor_master.csv', dtype=str)

print(f'Invoices : {inv_raw.shape}   Vendors : {vnd.shape}')

## 1. Overview

In [ ]:
print('--- invoices ---')
display(inv_raw.head(5))
print('\n--- vendor master ---')
display(vnd)

## 2. Completeness — Null / Empty Fields

In [ ]:
empty = (inv_raw == '').sum().rename('empty_count')
total = len(inv_raw)
completeness = pd.DataFrame({
    'empty_count': empty,
    'pct_missing': (empty / total * 100).round(1)
}).query('empty_count > 0').sort_values('pct_missing', ascending=False)

display(completeness)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(completeness.index, completeness['pct_missing'], color='#e07b54')
ax.set_xlabel('% missing')
ax.set_title('Missing values by column')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

## 3. Duplicates — Invoice Numbers

In [ ]:
dupes = inv_raw[inv_raw.duplicated('invoice_number', keep=False)].sort_values('invoice_number')
print(f'{dupes["invoice_number"].nunique()} duplicate invoice numbers ({len(dupes)} rows)')
display(dupes[['invoice_number', 'invoice_date', 'vendor_id', 'vendor_name', 'amount', 'approved_by']])

### 3b. Logical Duplicates — Same Vendor × Amount × Date (≤ 30 days)

In [ ]:
def _parse_date(d):
    for fmt in ['%Y-%m-%d', '%d.%m.%Y', '%d/%m/%Y', '%m-%d-%Y']:
        try:
            return pd.to_datetime(d, format=fmt)
        except ValueError:
            pass
    return pd.NaT

_amt_tmp = (
    inv_raw['amount']
    .str.replace('"', '', regex=False)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)
_date_tmp = inv_raw['invoice_date'].map(_parse_date)
_vid_tmp  = inv_raw['vendor_id'].str.replace(r'^V(\d)', r'V-\1', regex=True)

_work = inv_raw.assign(_amt=_amt_tmp, _date=_date_tmp, _vid=_vid_tmp).dropna(subset=['_date', '_amt'])

logical_dupes_pairs = []
for vid, grp in _work.groupby('_vid'):
    grp = grp.reset_index(drop=True)
    for i in range(len(grp)):
        for j in range(i + 1, len(grp)):
            r1, r2 = grp.iloc[i], grp.iloc[j]
            days = abs((r1['_date'] - r2['_date']).days)
            if r1['invoice_number'] != r2['invoice_number'] and r1['_amt'] == r2['_amt'] and days <= 30:
                logical_dupes_pairs.append({
                    'invoice_1': r1['invoice_number'], 'invoice_2': r2['invoice_number'],
                    'vendor':    r1['vendor_name'],
                    'amount':    r1['_amt'],
                    'date_1':    r1['_date'].date(), 'date_2': r2['_date'].date(),
                    'days_apart': days,
                })

logical_dupes_df = pd.DataFrame(logical_dupes_pairs)
print(f"{len(logical_dupes_df)} suspected logical duplicate pair(s) "
      f"(same vendor + amount, dates ≤ 30 days apart, different invoice number)")
if len(logical_dupes_df):
    display(logical_dupes_df)

## 4. Format Inconsistencies

### 4a. Date formats

In [ ]:
def detect_date_fmt(d):
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}', d):       return 'YYYY-MM-DD'
    if re.fullmatch(r'\d{2}\.\d{2}\.\d{4}', d):     return 'DD.MM.YYYY'
    if re.fullmatch(r'\d{2}/\d{2}/\d{4}', d):       return 'DD/MM/YYYY'
    if re.fullmatch(r'\d{2}-\d{2}-\d{4}', d):       return 'MM-DD-YYYY'
    return 'OTHER'

inv_raw['_date_fmt'] = inv_raw['invoice_date'].map(detect_date_fmt)
fmt_counts = inv_raw['_date_fmt'].value_counts()
print(fmt_counts.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
fmt_counts.plot.bar(ax=ax, color=['#5b8db8', '#e07b54', '#6abf85', '#c27dc0'])
ax.set_title('Date format distribution')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

### 4b. Currency casing

In [ ]:
bad_currency = inv_raw[inv_raw['currency'] != inv_raw['currency'].str.upper()]
print(f'{len(bad_currency)} rows with lowercase currency')
display(bad_currency[['invoice_number', 'vendor_name', 'amount', 'currency']])

### 4c. Vendor ID format (missing dash)

In [ ]:
bad_vid = inv_raw[inv_raw['vendor_id'].str.match(r'^V\d+$', na=False)]
print(f'{len(bad_vid)} rows with malformed vendor_id (e.g. V1002 instead of V-1002)')
display(bad_vid[['invoice_number', 'vendor_id', 'vendor_name', 'amount']])

### 4d. Vendor name inconsistencies (ALL CAPS / extra whitespace)

In [ ]:
name_issues = inv_raw[
    (inv_raw['vendor_name'].str.strip() != inv_raw['vendor_name']) |
    (inv_raw['vendor_name'].str.upper() == inv_raw['vendor_name'])
]
print(f'{len(name_issues)} rows with name casing / whitespace issues')
display(name_issues[['invoice_number', 'vendor_id', 'vendor_name']].drop_duplicates('vendor_name'))

### 4e. Amount formatting (embedded thousand-separator commas)

In [ ]:
comma_amounts = inv_raw[inv_raw['amount'].str.contains(r'^"', na=False)]
print(f'{len(comma_amounts)} rows with comma-formatted amounts')
display(comma_amounts[['invoice_number', 'vendor_name', 'amount', 'currency']])

## 5. Amount Validity — Negative Values

In [ ]:
inv = inv_raw.copy()
inv['amount_clean'] = (
    inv['amount']
    .str.replace('"', '', regex=False)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

negatives = inv[inv['amount_clean'] < 0]
print(f'{len(negatives)} invoices with negative amounts')
display(negatives[['invoice_number', 'invoice_date', 'vendor_name', 'description', 'amount_clean', 'approved_by']])

## 6. Referential Integrity — Invoices vs Vendor Master

In [ ]:
known_ids = set(vnd['vendor_id'])

# normalise vendor_id (fix missing dash)
inv['vendor_id_norm'] = inv['vendor_id'].str.replace(r'^V(\d)', r'V-\1', regex=True)

not_in_master = inv[
    (inv['vendor_id'] != '') & (~inv['vendor_id_norm'].isin(known_ids))
]
print(f'{len(not_in_master)} rows whose vendor_id does not match any master record (even after normalisation)')
if len(not_in_master):
    display(not_in_master[['invoice_number', 'vendor_id', 'vendor_id_norm', 'vendor_name']])

print('\nMissing vendor_id:')
display(inv[inv['vendor_id'] == ''][['invoice_number', 'vendor_name', 'amount_clean']])

## 7. Business Rules Violations

### 7a. Spend against INACTIVE or ON_HOLD vendors

In [ ]:
blocked = vnd[vnd['status'] != 'ACTIVE'][['vendor_id', 'vendor_name', 'status']]
print('Non-ACTIVE vendors:')
display(blocked)

inv_blocked = inv[inv['vendor_id_norm'].isin(blocked['vendor_id'])].copy()
inv_blocked = inv_blocked.merge(blocked, left_on='vendor_id_norm', right_on='vendor_id', suffixes=('_inv', '_master'))

summary = inv_blocked.groupby(['vendor_id_norm', 'vendor_name_master', 'status']).agg(
    invoice_count=('invoice_number', 'count'),
    total_spend=('amount_clean', 'sum')
).reset_index()

print(f'\n{len(inv_blocked)} invoices against blocked vendors — total exposure:')
display(summary)

### 7b. Invoices without PO number (no-PO spend)

In [ ]:
no_po = inv[inv['po_number'] == '']
no_po_spend = no_po['amount_clean'].sum()
total_spend  = inv['amount_clean'].sum()

print(f'{len(no_po)} invoices without PO ({len(no_po)/len(inv)*100:.1f}%)')
print(f'No-PO spend: {no_po_spend:,.0f}  ({no_po_spend/total_spend*100:.1f}% of total)')

by_cc = no_po.groupby('cost_center')['amount_clean'].agg(['count', 'sum']).rename(
    columns={'count': 'invoice_count', 'sum': 'spend'}
).sort_values('spend', ascending=False)
display(by_cc)

### 7c. Invoices without approver

In [ ]:
no_approver = inv[inv['approved_by'] == '']
print(f'{len(no_approver)} invoices with no approver ({len(no_approver)/len(inv)*100:.1f}%)')
display(no_approver[['invoice_number', 'invoice_date', 'vendor_name', 'amount_clean', 'cost_center']].sort_values('amount_clean', ascending=False).head(10))

### 7d. Approval Threshold Violations (Policy §3)

In [ ]:
THRESHOLDS = [
    (0,       5_000,        'Cost Center Owner'),
    (5_000,   50_000,       'Department Head'),
    (50_000,  250_000,      'Finance Director'),
    (250_000, float('inf'), 'CFO'),
]

def required_level(amount):
    if pd.isna(amount):
        return 'Unknown'
    for lo, hi, level in THRESHOLDS:
        if lo < amount <= hi:
            return level
    return 'Unknown'

inv['required_approver_level'] = inv['amount_clean'].map(required_level)

# Clear violations: > EUR 5k with no approver at all
threshold_violations = inv[
    (inv['amount_clean'] > 5_000) & (inv['approved_by'] == '')
][['invoice_number', 'vendor_name', 'amount_clean', 'required_approver_level', 'cost_center']]

# High-value invoices (> EUR 50k): approver present but level unverifiable from data — flag for manual review
high_value_flagged = inv[inv['amount_clean'] > 50_000][
    ['invoice_number', 'vendor_name', 'amount_clean', 'required_approver_level', 'approved_by']
].sort_values('amount_clean', ascending=False)

print(f"Invoices > EUR 5k with NO approver (policy §3.1 — clear violation): {len(threshold_violations)}")
display(threshold_violations)

print(f"\nInvoices > EUR 50k — require Finance Director or above (manual approver-level verification needed): {len(high_value_flagged)}")
display(high_value_flagged)

## 8. Quality Score Summary

In [ ]:
n = len(inv)

checks = {
    'Duplicate invoice numbers':                    len(dupes),
    'Logical duplicate pairs (content)':            len(logical_dupes_df),
    'Missing vendor_id':                            (inv['vendor_id'] == '').sum(),
    'Malformed vendor_id':                          len(bad_vid),
    'Vendor name casing/whitespace':                len(name_issues),
    'Lowercase currency':                           len(bad_currency),
    'Comma-formatted amounts':                      len(comma_amounts),
    'Negative amounts':                             len(negatives),
    'Non-standard date format (auto-corrected)':    (inv_raw['_date_fmt'] != 'YYYY-MM-DD').sum(),
    'No PO number':                                 len(no_po),
    'No approver':                                  len(no_approver),
    'Approval level violation (>5k, no approver)':  len(threshold_violations),
    'Spend on blocked vendors':                     len(inv_blocked),
}

# Issues that are pipeline-correctable — capped at orange regardless of volume
AUTO_CORRECTED = {'Non-standard date format (auto-corrected)'}

summary_df = pd.DataFrame.from_dict(checks, orient='index', columns=['affected_rows'])
summary_df['pct'] = (summary_df['affected_rows'] / n * 100).round(1)
summary_df = summary_df.sort_values('affected_rows', ascending=False)

display(summary_df)

score = max(0, 100 - summary_df['pct'].mean())
print(f'\nComposite quality score (indicative): {score:.0f} / 100')

colors = []
for label, row in summary_df.iterrows():
    if label in AUTO_CORRECTED:
        colors.append('#e8b84b')
    elif row['affected_rows'] > n * 0.1:
        colors.append('#d64e3b')
    elif row['affected_rows'] > 0:
        colors.append('#e8b84b')
    else:
        colors.append('#5fb36a')

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(summary_df.index, summary_df['pct'], color=colors)
ax.set_xlabel('% of invoice rows affected')
ax.set_title('Data quality issues — share of total invoice rows')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.invert_yaxis()
plt.tight_layout()
plt.show()